# Notebook 01 — MLB Stats API Data Pull

Source:`https://statsapi.mlb.com/api/v1`  
Rate limit: 0.5 s between calls  

This notebook pulls three datasets from the MLB Stats API for the 2025 season:

1. Players roster: all MLB players with ID, name, position, team  
2. Hitting stats: season batting lines for all hitters with ≥1 PA  
3. Pitching stats: season pitching lines for all pitchers with IP > 0  

The universal join key across all sources is `mlbam_id` (the MLBAM player ID).

In [2]:
import sys, os

# imports from project root
sys.path.insert(0, os.path.abspath(".."))

from scrapers.mlb_api import get_all_players, get_hitting_stats, get_pitching_stats
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

## 1. All Players Roster

Endpoint: `/sports/1/players?season=2025`

Returns every player on a 40-man roster for the given season. Fields: `mlbam_id`, `name`, `position`, `team`.

Quirks: Players who were traded mid-season show only their *current* team. Players on the 60-day IL are still included. The `team` field can be `None` for free agents.

In [4]:
players = get_all_players(2025)
print(f"Shape: {players.shape}")
print(f"\nDtypes:\n{players.dtypes}")
print(f"\nNull rates:\n{players.isnull().mean().round(4)}")
print(f"\nSample (10 rows):")
players.head(10)

Shape: (1470, 4)

Dtypes:
mlbam_id     int64
name        object
position    object
team        object
dtype: object

Null rates:
mlbam_id    0.0
name        0.0
position    0.0
team        0.0
dtype: float64

Sample (10 rows):


,mlbam_id,name,position,team
0,671096,Andrew Abbott,P,Cincinnati Reds
1,690953,Mick Abel,P,Minnesota Twins
2,691769,Philip Abner,P,Arizona Diamondbacks
3,682928,CJ Abrams,SS,Washington Nationals
4,650556,Bryan Abreu,P,Houston Astros
5,677800,Wilyer Abreu,RF,Boston Red Sox
6,691185,Maximo Acosta,SS,Miami Marlins
7,670183,Garrett Acton,P,Tampa Bay Rays
8,682668,Luisangel Acuña,2B,New York Mets
9,660670,Ronald Acuña Jr.,RF,Atlanta Braves


## 2. Hitting Stats

Endpoint: `/people?personIds={ids}&hydrate=stats(group=hitting,type=season,season=2025)`

Fetched in batches of 100 player IDs. Returns season batting stats for players with >= 1 PA. Deduplicated by `mlbam_id` (keeping the row with the most PA).

Quirks: 
- `AVG`, `OBP`, `SLG`, `OPS` come back as strings (e.g. `".310"`) — convert to float downstream.  
- `BB_pct` and `K_pct` are computed from `BB/PA` and `SO/PA`.

In [6]:
hitting = get_hitting_stats(2025)
print(f"Shape: {hitting.shape}")
print(f"\nDtypes:\n{hitting.dtypes}")
print(f"\nNull rates:\n{hitting.isnull().mean().round(4)}")
print(f"\nSample (10 rows):")
hitting.head(10)

Shape: (673, 13)

Dtypes:
mlbam_id          int64
PA                int64
AVG              object
OBP              object
SLG              object
OPS              object
HR                int64
RBI               int64
BB                int64
SO                int64
BB_pct          float64
K_pct           float64
primary_team     object
dtype: object

Null rates:
mlbam_id        0.0
PA              0.0
AVG             0.0
OBP             0.0
SLG             0.0
OPS             0.0
HR              0.0
RBI             0.0
BB              0.0
SO              0.0
BB_pct          0.0
K_pct           0.0
primary_team    0.0
dtype: float64

Sample (10 rows):


,mlbam_id,PA,AVG,OBP,SLG,OPS,HR,RBI,BB,SO,BB_pct,K_pct,primary_team
0,596019,732,.267,.346,.466,.812,31,86,65,131,8.9,17.9,New York Mets
1,646240,729,.252,.372,.479,.851,35,109,112,192,15.4,26.3,San Francisco Giants
2,660271,727,.282,.392,.622,1.014,55,102,109,187,15.0,25.7,Los Angeles Dodgers
3,656941,724,.240,.365,.563,.928,56,132,108,197,14.9,27.2,Philadelphia Phillies
4,621566,724,.272,.366,.484,.850,29,95,91,176,12.6,24.3,Atlanta Braves
5,672695,720,.290,.389,.462,.851,20,100,94,83,13.1,11.5,Arizona Diamondbacks
6,665742,715,.263,.396,.525,.921,43,105,127,137,17.8,19.2,New York Mets
7,677594,710,.267,.324,.474,.798,32,95,44,152,6.2,21.4,Seattle Mariners
8,668227,709,.238,.334,.426,.760,27,76,64,191,9.0,26.9,Seattle Mariners
9,624413,709,.272,.347,.524,.871,38,126,61,162,8.6,22.8,New York Mets


## 3. Pitching Stats

Endpoint: `/people?personIds={ids}&hydrate=stats(group=pitching,type=season,season=2025)`

Fetched in batches of 100 (filtered to position == "P" first). Returns season pitching stats for pitchers with IP > 0.

Quirks:
- `ERA` and `WHIP` come back as strings, convert to float downstream.  
- Per-9 rates (`K_per9`, `BB_per9`, `HR_per9`) are computed from raw counts / IP.  
- Some relievers may have very few IP, making per-9 rates noisy.

In [8]:
pitching = get_pitching_stats(2025)
print(f"Shape: {pitching.shape}")
print(f"\nDtypes:\n{pitching.dtypes}")
print(f"\nNull rates:\n{pitching.isnull().mean().round(4)}")
print(f"\nSample (10 rows):")
pitching.head(10)

Shape: (802, 13)

Dtypes:
mlbam_id          int64
GS                int64
G                 int64
IP              float64
ERA              object
WHIP             object
SO                int64
BB                int64
HR                int64
K_per9          float64
BB_per9         float64
HR_per9         float64
primary_team     object
dtype: object

Null rates:
mlbam_id        0.0
GS              0.0
G               0.0
IP              0.0
ERA             0.0
WHIP            0.0
SO              0.0
BB              0.0
HR              0.0
K_per9          0.0
BB_per9         0.0
HR_per9         0.0
primary_team    0.0
dtype: float64

Sample (10 rows):


,mlbam_id,GS,G,IP,ERA,WHIP,SO,BB,HR,K_per9,BB_per9,HR_per9,primary_team
0,657277,34,34,207.0,3.22,1.24,224,46,14,9.74,2.00,0.61,San Francisco Giants
1,676979,32,32,205.1,2.59,1.03,255,46,24,11.19,2.02,1.05,Boston Red Sox
2,650911,32,32,202.0,2.50,1.06,212,44,12,9.45,1.96,0.53,Philadelphia Phillies
3,669373,31,31,195.1,2.21,0.89,241,33,18,11.12,1.52,0.83,Detroit Tigers
4,608331,32,32,195.1,2.86,1.10,189,51,14,8.72,2.35,0.65,New York Yankees
5,607074,33,33,195.1,3.09,1.05,203,73,22,9.36,3.37,1.01,New York Yankees
6,592332,32,32,193.0,3.59,1.06,189,50,21,8.81,2.33,0.98,Toronto Blue Jays
7,668678,33,33,192.0,4.83,1.26,175,66,31,8.20,3.09,1.45,Arizona Diamondbacks
8,664285,31,31,192.0,3.66,1.24,187,68,15,8.77,3.19,0.70,Houston Astros
9,694973,32,32,187.2,1.97,0.95,216,42,11,10.38,2.02,0.53,Pittsburgh Pirates


## 4. Null Rate Summary (all three DataFrames)


In [10]:
null_summary = pd.DataFrame({
    "players": players.isnull().mean().round(4),
    "hitting": hitting.isnull().mean().round(4),
    "pitching": pitching.isnull().mean().round(4),
}).fillna("—")
print("Null Rate Summary:")
null_summary

Null Rate Summary:


,players,hitting,pitching
AVG,—,0.0,—
BB,—,0.0,0.0
BB_pct,—,0.0,—
BB_per9,—,—,0.0
ERA,—,—,0.0
G,—,—,0.0
GS,—,—,0.0
HR,—,0.0,0.0
HR_per9,—,—,0.0
IP,—,—,0.0


## 5. Save to Parquet

In [12]:
data_dir = os.path.join("..", "data")
os.makedirs(data_dir, exist_ok=True)

players.to_parquet(os.path.join(data_dir, "raw_mlb_api_players.parquet"), index=False)
hitting.to_parquet(os.path.join(data_dir, "raw_mlb_api_hitting.parquet"), index=False)
pitching.to_parquet(os.path.join(data_dir, "raw_mlb_api_pitching.parquet"), index=False)

print("Saved:")
for f in ["raw_mlb_api_players.parquet", "raw_mlb_api_hitting.parquet", "raw_mlb_api_pitching.parquet"]:
    path = os.path.join(data_dir, f)
    size_kb = os.path.getsize(path) / 1024
    print(f"  {f} — {size_kb:.1f} KB")

Saved:
  raw_mlb_api_players.parquet — 31.3 KB
  raw_mlb_api_hitting.parquet — 29.1 KB
  raw_mlb_api_pitching.parquet — 32.2 KB
